In [ ]:
# Build the persistent ChromaDB vector store used by the semantic search service.

# This follows the exact patten taught in the labs/class. 

# The one difference from the labs is the storage type... the assignment asks for a ChromaDB PersistentClient instead of 
# the in-memory client used in labs. Persistent storage means we embed the data once, save it, and the app just reloads it.

# Dataset: a list of 150 cities ranked by livability using a variety of indicators ('livable_cities.csv').
# It has 12 columns, including the rank, city, country, and 9 indicators

# Run this once (the graders do not need to): `python build_index.py`

In [21]:
import re 

import chromadb
import pandas as pd
import time

from config import CHROMA_DIR, COLLECTION_NAME, DATA_CSV, EMBEDDING_MODEL, client

SAMPLE_SIZE = 150

# The columns we use from the UNESCO CSV.
RANK_COL = "Rank" 
CITY_COL = "City"   
COUNTRY_COL = "Country"
QUALITYLIFE_COL = "Quality of Life Index"
PURCHASINGPOWER_COL = "Purchasing Power Index"
SAFETY_COL = "Safety Index"
HEALTHCARE_COL = "Health Care Index"
COSTLIVING_COL = "Cost of Living Index"
PROPERTYPRICETOINCOME_COL = "Property Price to Income Ratio"
COMMUTETIME_COL = "Traffic Commute Time Index"
POLLUTION_COL = "Pollution Index"
CLIMATE_COL = "Climate Index"
DESCRIPTION_COL = "Description"

In [22]:
def embed_in_batches(texts, batch_size=10, max_retries=6):
    """Embed a list of texts in small chunks, retrying with longer waits if the
    gateway rate-limits us."""
    all_embeddings = []
    for start in range(0, len(texts), batch_size):
        batch = texts[start:start + batch_size]
        for attempt in range(max_retries):
            try:
                response = client.embeddings.create(input=batch, model=EMBEDDING_MODEL)
                all_embeddings.extend(item.embedding for item in response.data)
                break                                   # success -> next batch
            except Exception:
                wait = 5 * (attempt + 1)                # 5s, 10s, 15s, ...
                print(f"  rate limited at {start}; waiting {wait}s "
                      f"(attempt {attempt + 1}/{max_retries})")
                time.sleep(wait)
        else:
            raise RuntimeError(f"Gave up on batch starting at {start}")
        print(f"  embedded {start + len(batch)}/{len(texts)}")
        time.sleep(1)                                   # pause between batches
    return all_embeddings

In [23]:
def build():
    # 1) Load the dataset with pandas (same as the labs).
    df = pd.read_csv(DATA_CSV)

    # 2) Build, for each city, the text we will embed + the metadata we keep.
    documents = []   # the text that gets turned into an embedding
    ids = []         # a unique id per document (required by Chroma)
    metadatas = []   # extra fields we can display or filter on later

    for i, row in df.iterrows():
        city = str(row[CITY_COL])
        country = str(row[COUNTRY_COL])
        rank = int(row[RANK_COL])
        qualitylife = float(row[QUALITYLIFE_COL])
        purchasingpower = float(row[PURCHASINGPOWER_COL])
        safety = float(row[SAFETY_COL])
        healthcare = float(row[HEALTHCARE_COL])
        costliving = float(row[COSTLIVING_COL])
        propertypricetoincome = float(row[PROPERTYPRICETOINCOME_COL])
        commutetime = float(row[COMMUTETIME_COL])
        pollution = float(row[POLLUTION_COL])
        climate = float(row[CLIMATE_COL])
        description = str(row[DESCRIPTION_COL])


        # The document combines the city name, the country and the description.
        document = f"{city} ({country}): {description}"

        documents.append(document)
        ids.append(f"site_{i}")          # row index guarantees a unique id
        metadatas.append(
            {
                "city": city,
                "country": country,
                "rank": rank,
                "qualitylife": qualitylife,
                "purchasingpower": purchasingpower,
                "safety": safety,
                "healthcare": healthcare,
                "costliving": costliving,
                "propertypricetoincome": propertypricetoincome,
                "commutetime": commutetime,
                "pollution": pollution,
                "climate": climate,
                "description": description
            }
        )

    # 3) Turn each document into an embedding vector, batched to avoid rate limits.
    print("Creating embeddings in batches...")
    embeddings = embed_in_batches(documents, batch_size=10)

    # 4) Save everything into a PersistentClient so it is written to ./chroma_db
    #    and can be reloaded by the app without re-embedding.
    chroma_client = chromadb.PersistentClient(path=CHROMA_DIR)
    try:
        chroma_client.delete_collection(COLLECTION_NAME)   # start fresh on rebuild
    except Exception:
        pass
    collection = chroma_client.create_collection(name=COLLECTION_NAME)

    # Add the documents, their embeddings, ids, and metadata (as in the labs).
    collection.add(
        ids=ids,
        documents=documents,
        metadatas=metadatas,
        embeddings=embeddings,
    )
    print(f"Indexed {collection.count()} sites into {CHROMA_DIR}")


if __name__ == "__main__":
    build()

Creating embeddings in batches...
  embedded 10/150
  embedded 20/150
  embedded 30/150
  embedded 40/150
  embedded 50/150
  embedded 60/150
  embedded 70/150
  embedded 80/150
  embedded 90/150
  embedded 100/150
  embedded 110/150
  embedded 120/150
  embedded 130/150
  embedded 140/150
  embedded 150/150
Indexed 150 sites into c:\Users\ziral\Documents\MSc_PhD Nutritional Science\Data Science Institute Certificate\Assignments\deploying-ai\05_src\assignment_2_chat\chroma_db
